# Implementing Uncertainty Quantification for Robotic Failure Detection

---

# Table of Content

1. Classifier in Robotics (Classifying if Jenga tower is toppled or not)
2. Uncertainty Quantification (UQ)
    - Certainty of a single prediction (Baseline)
    - Ensemble Variance
    - Monte Carlo Dropout
    - Empirical Distribution of Features (Density Estimation)
3. Calibration of the Uncertainty Threshold for Detecting Out-of-Distribution Data
    - Conformal Prediction
    - Trajectory-level Conformal Prediction for Sequential Robotics Data.

In [ ]:
!git clone https://github.com/junwon-vision/eais_uq.git
import os
import sys
from pathlib import Path

EAIS_UQ_DIR = Path("eais_uq").resolve()
if str(EAIS_UQ_DIR) not in sys.path:
    sys.path.insert(0, str(EAIS_UQ_DIR))

CKPT_ZIP = Path("model_checkpoints.zip")
CKPT_DIR = Path("checkpoints")

import gdown
gdown.download(id="1d9W43ejh3Fv98tmnwhCBqZVYms86U_0U", output=str(CKPT_ZIP), quiet=False)
!unzip -q -o {CKPT_ZIP} -d .
print("Checkpoints extracted.")

!mv data eais_uq/data

Load dataset and pretrained models!

In [ ]:
import sys, os, pickle, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import Video

# Local imports (self-contained package)
from config import *
from data_utils import *
from models import *
from uq_inference import *
from density import *
from conformal import *
from visualization import *

print(f"Device: {DEVICE}")
print(f"Data dir: {DATA_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"OOD dir: {OOD_DIR}")

# ── Load ALL models once (shared across all subsequent cells) ──
print("\nLoading models...")
baseline_model, _  = load_classifier(CLASSIFIER_CHECKPOINTS["baseline"],   DEVICE, mode="baseline")
mc_model,       _  = load_classifier(CLASSIFIER_CHECKPOINTS["mc_dropout"], DEVICE, mode="mc_dropout")
ensemble_model, _  = load_classifier(CLASSIFIER_CHECKPOINTS["ensemble"],   DEVICE, mode="ensemble")
density_model, dino_encoder = load_density_model(DENSITY_CHECKPOINT, DEVICE)
print("All models loaded.")

# ── Load datasets once ──
episodes = load_episodes(["failure_labeled.pkl"])
eval_episodes = load_episodes(["failure_eval.pkl"])
cal_episodes = load_episodes(["calibration_combined.pkl"])
print(f"Training episodes: {len(episodes)}, Eval: {len(eval_episodes)}, Cal: {len(cal_episodes)}")

# ── Load OOD trajectories once ──
OOD_EP_INDICES = [0, 1, 2, 8, 10]

def load_ood_episode(idx, ood_dir=OOD_DIR):
    """Load one OOD trajectory pkl; return episode dict with raw numpy images."""
    pkl = sorted(ood_dir.glob("traj_*.pkl"))[idx]
    with open(pkl, "rb") as f:
        data = pickle.load(f)
    traj = data[0]
    front_imgs, wrist_imgs, labels = [], [], []
    for info, _, label in traj:
        front_imgs.append(info["cam_rs"][0])
        wrist_imgs.append(info["cam_zed_right"][0])
        labels.append(label)
    return {"front_cam": front_imgs, "wrist_cam": wrist_imgs,
            "failure": np.array(labels, dtype=np.float32)}

ood_episodes = [load_ood_episode(i) for i in OOD_EP_INDICES]
print(f"OOD episodes loaded: {len(ood_episodes)}")

## 1. What is Uncertainty Quantification?

A predictive model produces predictions $\hat{y} = f_\theta(x)$, but how confident should we be in that prediction?

**Uncertainty Quantification (UQ)** answers this question by treating the prediction as a random variable and characterizing its distribution.

### Bayesian Formulation

Given training data $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^N$, the **Bayesian predictive distribution** is:

$$p(y \mid x, \mathcal{D}) = \int p(y \mid x, \theta)\, p(\theta \mid \mathcal{D})\, d\theta$$

where:
- $p(\theta \mid \mathcal{D})$ is the **posterior** over model parameters
- $p(y \mid x, \theta)$ is the **likelihood** (model prediction for fixed parameters)

**(Side note) Decomposition of Uncertainty**

Variance provides a practical measure of uncertainty, as it captures the spread of a distribution. Moreover, this predictive uncertainty can be decomposed into aleatoric and epistemic components.

By the **law of total variance**, the predictive variance decomposes into:

$$\underbrace{\text{Var}[y \mid x, \mathcal{D}]}_{\text{total}} = \underbrace{\mathbb{E}_\theta[\text{Var}[y \mid x, \theta]]}_{\text{aleatoric}} + \underbrace{\text{Var}_\theta[\mathbb{E}[y \mid x, \theta]]}_{\text{epistemic}}$$

| Type | Source | Reducible? |
|------|--------|------------|
| **Aleatoric** | Inherent noise in the data | No (irreducible) |
| **Epistemic** | Limited training data / model capacity | Yes (with more data) |

---


## 2. Problem Setup: Binary Failure Classification

We study a **binary image classifier** that predicts whether a robotic Jenga stacking attempt is **safe** or a **failure (tower toppled / block fallen)**.

Formally, we learn a classifier for the conditional distribution $p(y \mid x, \theta)$, where:
- $x$ is a pair of images (front-view + wrist-view) of the scene
- $y \in \{0, 1\}$ is the label, with 0 = safe and 1 = failure

**Concrete example: robotic Jenga block stacking**
- Given two camera views of the Jenga tower after an action, the classifier must decide **from the images alone** whether the tower has toppled (failure) or is still standing (safe).

**Model setup:**
- **Input** $x$: two camera images (front view + wrist view), each 256×256 RGB
- **Output** $y \in \{0, 1\}$: 0 = safe, 1 = failure (tower toppling / block falling)
- **Model**: frozen DINOv2 ViT-S/14 backbone → MLP head

### 2.1 Inspect the Dataset

Training: Used 92 labeled episodes.
Label: 0 = safe, 1 = failure.

The model is already trained. You can optionally train your own model!

In [ ]:
# Dataset statistics (episodes loaded in setup cell)
print(f"Total episodes: {len(episodes)}")

# Print statistics
total_steps = 0
n_success, n_failure, n_unknown = 0, 0, 0
lengths = []
for ep in episodes:
    fail_arr = ep["failure"]
    T = len(fail_arr)
    lengths.append(T)
    total_steps += T
    n_success += int((fail_arr == 0).sum())
    n_failure += int((fail_arr == 1).sum())
    n_unknown += int((fail_arr == -1).sum())

lengths = np.array(lengths)
print(f"\nTotal timesteps: {total_steps}")
print(f"Trajectory lengths: min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.1f}")
print(f"\nLabel distribution:")
print(f"  Success (0):  {n_success:>7} ({100*n_success/total_steps:.1f}%)")
print(f"  Failure (1):  {n_failure:>7} ({100*n_failure/total_steps:.1f}%)")
print(f"  Unknown (-1): {n_unknown:>7} ({100*n_unknown/total_steps:.1f}%)")
print(f"  After mapping (-1 → 0): Success={n_success+n_unknown}, Failure={n_failure}")

Visualize a sample image and episode video from the dataset.

In [ ]:
# Load sample images and episode video from media/
from pathlib import Path
from IPython.display import Video

media = Path("media")
img_3rd = plt.imread(str(media / "dataset_sample_3rd.jpg"))
img_wrist = plt.imread(str(media / "dataset_sample_wrist.jpg"))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_3rd)
axes[0].set_title("3rd-person (front cam)")
axes[0].axis("off")
axes[1].imshow(img_wrist)
axes[1].set_title("Wrist cam")
axes[1].axis("off")
plt.tight_layout()
plt.show()

Video("media/episode_labels_ep0000_T76.mp4", embed=True)

### 2.2 Model Architecture

The failure classifier uses a **frozen DINOv2 ViT-S/14** backbone (384-dim CLS token) for each camera view. The features are concatenated and passed through a trainable MLP:

```
front_cam (256×256) → resize 224 → DINOv2 (frozen) → 384-dim ┐
wrist_cam (256×256) → resize 224 → DINOv2 (frozen) → 384-dim ┘ concat → 768-dim
                                                                    ↓
                                                Linear(768, 512) + ReLU + Dropout
                                                Linear(512, 256) + ReLU + Dropout
                                                Linear(256, 128) + ReLU + Dropout
                                                Linear(128, 128) + ReLU + Dropout
                                                Linear(128, 1) → logit → σ(·) → P(failure)
```

In [ ]:
# Print architecture of the ensemble model (loaded in setup cell)
print("Single member architecture:")
print(ensemble_model.members[0].head)
print(f"\nEnsemble size: {len(ensemble_model.members)}")

total_params = sum(p.numel() for p in ensemble_model.members[0].parameters())
trainable_params = sum(p.numel() for p in ensemble_model.members[0].parameters() if p.requires_grad)
print(f"Total parameters per member: {total_params:,}")
print(f"Trainable parameters (head only): {trainable_params:,}")
print(f"Frozen parameters (DINOv2 backbone): {total_params - trainable_params:,}")

### 2.3 Baseline Prediction

Let's run the baseline (single deterministic forward pass) on a training trajectory.

In [ ]:
# Baseline prediction on labeled dataset (models & data loaded in setup cell)
print(f"Eval episodes: {len(episodes)}")

# Pick one trajectory with failure, one without
idx_with_failure = next(i for i, ep in enumerate(episodes) if (np.array(ep["failure"]) == 1).any())
idx_no_failure = next(i for i, ep in enumerate(episodes) if (np.array(ep["failure"]) != 1).all())
ep_failure = episodes[idx_with_failure]
ep_success = episodes[idx_no_failure]

# Trajectory with failure
probs_f, _, ent_f = run_inference_episode(baseline_model, ep_failure, mode="baseline", device=DEVICE)
path_f = render_episode_video_no_graph(
    ep_failure, idx_with_failure, probs_f, ent_f,
    uq_label="Entropy", mode="baseline", fps=10,
    output_path=OUTPUT_DIR / "videos" / "baseline_with_failure.mp4")
print(f"With failure (ep {idx_with_failure}): {path_f}")
display_video(path_f)

# Trajectory without failure
probs_s, _, ent_s = run_inference_episode(baseline_model, ep_success, mode="baseline", device=DEVICE)
path_s = render_episode_video_no_graph(
    ep_success, idx_no_failure, probs_s, ent_s,
    uq_label="Entropy", mode="baseline", fps=10,
    output_path=OUTPUT_DIR / "videos" / "baseline_no_failure.mp4")
print(f"Without failure (ep {idx_no_failure}): {path_s}")
display_video(path_s)

---

## 3. Uncertainty Quantification Methods

We now explore four practical approaches to quantifying model uncertainty. For each, we present the mathematical formulation, the implementation, and a demo on an evaluation trajectory.

### 3.1 Entropy of Predictive Distribution

The simplest UQ measure: the **binary entropy** of the model's output probability.

$$H(p) = \mathbb{E}\big[-\log p(y | x)\big] = -p \log p - (1-p) \log(1-p) \approx 1 - \argmax_c p(c)$$

where $p = p(y=1 | x)$ is the predicted failure probability.

**Key property:** $H$ is maximized at $p = 0.5$ (maximum uncertainty) and minimized at $p \in \{0, 1\}$ (full confidence).

- in general, entropy is maximized with uniform distribution, while it is maximized with a certain prediction (e.g., dirac distribution).

**Limitation:** Entropy only captures *uncertainty in the output*, not *uncertainty about the model*. A confidently wrong model has low entropy.

### Visualize the prediction & uncertainty.

- <span style="color:green">Green border</span>: Safe
- <span style="color:red">Red border</span>: Failure
- Bottom Graph: Entropy

In [ ]:
EVAL_EP_INDICES = [1]

base_results = {}
for idx in EVAL_EP_INDICES:
    base_results[idx] = run_inference_episode(
        baseline_model, eval_episodes[idx], mode="baseline", device=DEVICE)
base_probs, base_vars, base_ents = base_results[EVAL_EP_INDICES[0]]

VIDEO_BASE = OUTPUT_DIR / "videos" / "baseline_ep0.mp4"
video_path = render_episode_video(
    eval_episodes[0], 0, base_probs, base_ents,
    uq_label="Entropy", mode="baseline", fps=10,
    uq_ymax=MODE_UQ_YMAX["baseline"],
    output_path=VIDEO_BASE)
print(f"Video saved: {video_path}")
display_video(video_path)

The uncertainty value is high when the model's prediction is wrong. (e.g., at timestep 100)

Q: What is the limitation of the prediction confidence?

### 3.2 Deep Ensemble

**Key idea** (Lakshminarayanan et al., 2017): Train $K$ independent models with different random initializations. The ensemble approximates the posterior as a mixture of point masses:

$$p(\theta \mid \mathcal{D}) \approx \frac{1}{K} \sum_{k=1}^{K} \delta(\theta - \theta_k)$$

$$p(y \mid x, \mathcal{D}) = \int p(y \mid x, \theta)\, p(\theta \mid \mathcal{D})\, d\theta \approx
\frac{1}{K}\sum_{k=1}^{K} p(y \mid x, \theta_k)$$

**Inference:**

$$
\bar{p}(y \mid x)
:=
\frac{1}{K}\sum_{k=1}^{K} p(y \mid x, \theta_k)
$$

$$
\mathrm{Var}\big[p(y \mid x,\theta)\big]
\approx
\frac{1}{K}\sum_{k=1}^{K}
\left(
p(y \mid x,\theta_k)-\bar{p}(y \mid x)
\right)^2
$$

In our implementation, all $K$ members share the frozen DINOv2 backbone but have **independently** initialized MLP heads.

```python
class FailureClassifierDINOv2Ensemble(nn.Module):
    """Ensemble of K FailureClassifierDINOv2 instances with shared backbone."""

    def __init__(self, ensemble_size=5, dropout_rate=0.2, freeze_backbone=True):
        super().__init__()
        self.members = nn.ModuleList()
        for i in range(ensemble_size):
            member = FailureClassifierDINOv2(dropout_rate, freeze_backbone)
            init_head_weights(member.head, seed=i * 9999 + 7)
            self.members.append(member)

    def forward(self, front, wrist):
        """Returns stacked logits: (K, B)."""
        return torch.stack([m(front, wrist) for m in self.members], dim=0)

def init_head_weights(head, seed=None):
    """Initialize MLP head with Kaiming init. Different seed per ensemble member."""
    if seed is not None:
        torch.manual_seed(seed)
    for m in head.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            nn.init.zeros_(m.bias)

# Ensemble prediction
logits_np = all_logits.cpu().numpy()
probs_np = 1.0 / (1.0 + np.exp(-logits_np))
mean_probs = probs_np.mean(axis=0)
variances = logits_np.var(axis=0) if logit_variance else probs_np.var(axis=0)
```

In [ ]:
EVAL_EP_INDICES = [0]
ens_results = {}
for idx in EVAL_EP_INDICES:
    ens_results[idx] = run_inference_episode(
        ensemble_model, eval_episodes[idx], mode="ensemble", device=DEVICE)
ens_probs, ens_vars, ens_ents = ens_results[EVAL_EP_INDICES[0]]

VIDEO_ENS = OUTPUT_DIR / "videos" / "ensemble_ep0.mp4"
video_path = render_episode_video(
    eval_episodes[0], 0, ens_probs, ens_vars,
    uq_label="Variance", mode="ensemble", fps=10,
    uq_ymax=MODE_UQ_YMAX["ensemble"],
    output_path=VIDEO_ENS)
print(f"Video saved: {video_path}")
display_video(video_path)

Q: What is the limitation of the ensemble?

---

### 3.3 Monte Carlo Dropout

**Key idea** (Gal & Ghahramani, 2016): Dropout at test time approximates variational Bayesian inference. Each stochastic forward pass samples from an approximate posterior.

**Training:** Standard training with dropout rate $p_{\text{drop}}$.

**Inference:** Keep dropout **enabled** and run $T$ forward passes:

$$\bar{p}(y \mid x)
:=
\frac{1}{T}\sum_{t=1}^{T} p(y \mid x,\theta_t)
$$

$$
\mathrm{Var}\big[p(y \mid x,\theta)\big]
\approx
\frac{1}{T}\sum_{t=1}^{T}
\left(
p(y \mid x,\theta_t)-\bar{p}(y \mid x)
\right)^2$$

where $\theta_t$ denotes the network with a different dropout mask at each pass.

**Implementation:**
```python
def enable_mc_dropout(model):
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()  # keep dropout active
```

In [ ]:
EVAL_EP_INDICES = [0]
mc_results = {}
for idx in EVAL_EP_INDICES:
    mc_results[idx] = run_inference_episode(
        mc_model, eval_episodes[idx], mode="mc_dropout", device=DEVICE)
mc_probs, mc_vars, mc_ents = mc_results[EVAL_EP_INDICES[0]]

# Video for episode 0
VIDEO_MC = OUTPUT_DIR / "videos" / "mc_dropout_ep0.mp4"
video_path = render_episode_video(
    eval_episodes[0], 0, mc_probs, mc_vars,
    uq_label="Variance", mode="mc_dropout", fps=10,
    uq_ymax=MODE_UQ_YMAX["mc_dropout"],
    output_path=VIDEO_MC)
print(f"Video saved: {video_path}")
display_video(video_path)

Q: What is the limitation of MC Dropout?

---

### 3.4 Density Estimation (Flow Matching)

**Key idea:** Instead of modeling $p(\theta \mid \mathcal{D})$, model the **empirical feature distribution** $p(z)$ where $z = \text{DINOv2}(x)$. Points with low feature density are likely out-of-distribution.

Unlike Bayesian predictive methods such as deep ensembles or MC dropout, which aim to estimate the **predictive distribution**
$$
p(y \mid x, \mathcal{D}),
$$
density estimation does **not** directly model uncertainty over the label $y$. Instead, it models how likely the input itself is under the training data distribution, i.e.,
$$
p(x \mid \mathcal{D})
\quad\text{or, more practically in our setup,}\quad
p(z \mid \mathcal{D}),
$$
where $z$ is the learned feature representation of $x$.


**Practical Instantiation: Flow matching** learns a vector field $v_\theta(x_t, t)$ that maps the data distribution to a standard Gaussian:

$$x_t = x_0 + t\,(x_1 - x_0), \quad x_0 \sim p_{\text{data}}, \quad x_1 \sim \mathcal{N}(0, I), \quad t \sim \mathcal{U}(0,1)$$

$$\mathcal{L} = \mathbb{E}\big[\|v_\theta(x_t, t) - (x_1 - x_0)\|^2\big]$$

**Inference** (density proxy — following [FailDetect, arXiv:2503.08558](https://arxiv.org/abs/2503.08558)):

$$z = x + v_\theta(x, t{=}0), \quad \text{logpZO} = \|z\|^2$$

Higher $\text{logpZO}$ → feature is far from training distribution → more uncertain.

**Key distinction:** This measures *distributional* uncertainty, not *posterior* uncertainty. It answers "is this input similar to training data?" rather than "is the model uncertain about this input?".

#### Training the Density Estimator

The flow matching model is trained to map the empirical DINOv2 feature distribution to a standard Gaussian. Below is a simplified training loop:

```python
# Extract DINOv2 features from training images
features = []  # shape: (N, 384)
for ep in train_episodes:
    for front, wrist in zip(ep["front_cam"], ep["wrist_cam"]):
        z = dino_encoder(preprocess(front), preprocess(wrist))  # (1, 768)
        features.append(z)
features = torch.cat(features, dim=0)  # (N, 768)

# Training loop
for epoch in range(num_epochs):
    x0 = sample_batch(features)            # data samples
    x1 = torch.randn_like(x0)              # noise samples
    t = torch.rand(batch_size, 1)           # random timesteps
    xt = x0 + t * (x1 - x0)                # interpolated points
    target = x1 - x0                        # target velocity
    pred = model(xt, t)                     # predicted velocity
    loss = F.mse_loss(pred, target)
    loss.backward()
    optimizer.step()
```

> **Note on interpretation:** The `logpZO` value represents the **negative log-likelihood** in our implementation. **Higher `logpZO` values indicate greater OOD-ness** (lower density under the training distribution). In other words, the likelihood is *inversely proportional* to the `logpZO` score: $p(z) \propto e^{-\text{logpZO}}$.

Run density estimation on eval episodes.

Q: Is the density-based uncertainty measure relates to the prediction accuracy?

Q: What is the limitation of the density estimation?

Q: What is the assumption of the density estimation?

In [ ]:
# Density estimation on eval episodes (models loaded in setup cell)
EVAL_EP_INDICES = [0]

logpZO_results = {}
for idx in EVAL_EP_INDICES:
    logpZO_results[idx] = predict_density_episode(
        density_model, dino_encoder, eval_episodes[idx], DEVICE)
logpZO = logpZO_results[EVAL_EP_INDICES[0]]

VIDEO_DENSITY = OUTPUT_DIR / "videos" / "density_ep0.mp4"
video_path = render_episode_video_density(
    eval_episodes[0], 0, ens_probs, logpZO, ens_vars,
    uq_label="Variance", mode="ensemble", fps=10,
    uq_ymax=MODE_UQ_YMAX["ensemble"],
    logpZO_ymax=DENSITY_YMAX,
    output_path=VIDEO_DENSITY)
print(f"Video saved: {video_path}")
display_video(video_path)

## 4. Comparing UQ Methods

In [ ]:
# Side-by-side UQ comparison (uses models loaded in setup cell)
ep = eval_episodes[1]
T = len(ep["failure"])
gt = np.array(ep["failure"])

# Run inference on this episode
_, _, cmp_base_ents = run_inference_episode(baseline_model, ep, mode="baseline",   device=DEVICE)
_, cmp_mc_vars,   _ = run_inference_episode(mc_model,       ep, mode="mc_dropout", device=DEVICE)
_, cmp_ens_vars,  _ = run_inference_episode(ensemble_model, ep, mode="ensemble",   device=DEVICE)
cmp_dens = predict_density_episode(density_model, dino_encoder, ep, DEVICE)

# Plot UQ scores only (no P(fail))
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)

methods = [
    ("Baseline (Entropy)",    cmp_base_ents, "darkblue", MODE_UQ_YMAX["baseline"]),
    ("MC Dropout (Variance)", cmp_mc_vars,   "purple",   MODE_UQ_YMAX["mc_dropout"]),
    ("Ensemble (Variance)",   cmp_ens_vars,  "green",    MODE_UQ_YMAX["ensemble"]),
    ("Density (logpZO)",      cmp_dens,      "darkgreen", DENSITY_YMAX),
]

for ax, (name, uq_m, color, ymax) in zip(axes, methods):
    ax.plot(range(T), uq_m, color=color, linewidth=1.5, label="UQ")
    ax.fill_between(range(T), 0, uq_m, color=color, alpha=0.1)
    ax.set_ylabel(name.split("(")[1].rstrip(")"), fontsize=8, color=color)
    ax.set_ylim(0, ymax)
    if (gt == 1).any():
        ax.fill_between(range(T), 0, 0.02 * ymax, where=gt == 1, color='red', alpha=0.5)
    ax.set_title(name, fontsize=10)

axes[-1].set_xlabel("Timestep")
fig.suptitle("Side-by-Side UQ Comparison — eval_episodes[1]", fontsize=12)
fig.tight_layout()
plt.show()

Compare the UQ methods with g.t. labels.
Uncertainty is higher when the model's prediction is wrong.

In [ ]:
# Quantitative comparison on eval set
from sklearn.metrics import accuracy_score, roc_auc_score

EVAL_EP_INDICES = [0, 1, 3, 9, 12]

print("Running quantitative evaluation...")
methods_eval = {
    "baseline": (baseline_model, "baseline", None),
    "mc_dropout": (mc_model, "mc_dropout", None),
    "ensemble": (ensemble_model, "ensemble", None),
}

eval_eps_subset = [eval_episodes[i] for i in EVAL_EP_INDICES]
for name, (m, mode, lap) in methods_eval.items():
    all_probs, all_vars, all_ents, all_labels = [], [], [], []
    for ep in eval_eps_subset:
        p, v, e = run_inference_episode(m, ep, mode=mode, device=DEVICE, laplace=lap)
        labels = np.array(ep["failure"], dtype=np.float32)
        labels[labels < 0] = 0.0
        all_probs.extend(p)
        all_vars.extend(v)
        all_ents.extend(e)
        all_labels.extend(labels)

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_vars = np.array(all_vars)
    all_ents = np.array(all_ents)
    preds = (all_probs > 0.5).astype(float)
    correct = preds == all_labels

    acc = accuracy_score(all_labels, preds)

    # UQ quality: baseline uses entropy, others use variance
    uq_vals = all_ents if mode == "baseline" else all_vars
    uq_name = "Entropy" if mode == "baseline" else "Variance"
    uq_correct = uq_vals[correct].mean() if correct.sum() > 0 else float('nan')
    uq_wrong = uq_vals[~correct].mean() if (~correct).sum() > 0 else float('nan')

    print(f"  {name:12s} | Acc={acc:.4f}  | "
          f"{uq_name}(correct)={uq_correct:.6f}  {uq_name}(wrong)={uq_wrong:.6f}  "
          f"ratio={uq_wrong/(uq_correct+1e-10):.2f}x")

## 4. OOD Evaluation

**How does the model prediction & uncertainty estimation perform on OOD inputs?**

Is andrea safe?
Is andrea OOD?

In [ ]:
# ── 4.1  Single OOD image: andrea.png ───────────────────────
# EVAL_EP_INDICES = [0, 1, 3, 9, 12]
EVAL_EP_INDICES = [0, 1]

from data_utils import _img_transform

def _infer_single_image(img_np):
    """Run all UQ methods on a single (256,256,3) uint8 RGB numpy image.
    Uses the same image for both front and wrist cameras.
    """
    front_t = _img_transform(img_np).unsqueeze(0).to(DEVICE)
    wrist_t = front_t

    with torch.no_grad():
        logit = baseline_model(front_t, wrist_t)
        p = torch.sigmoid(logit).item()
        ent = -(p * np.log(p + 1e-8) + (1 - p) * np.log(1 - p + 1e-8))

        mc_logits = mc_dropout_predict(mc_model.members[0], front_t, wrist_t, T=50)
        mc_p = torch.sigmoid(mc_logits).cpu().numpy().squeeze()
        mc_var = float(mc_p.var())

        ens_logits = ensemble_model(front_t, wrist_t)
        ens_p = torch.sigmoid(ens_logits).cpu().numpy().squeeze()
        ens_var = float(ens_p.var())

        mini_ep = {"front_cam": [img_np], "wrist_cam": [img_np], "failure": [0]}
        logpZO_arr = predict_density_episode(density_model, dino_encoder, mini_ep, DEVICE)
        dens_val = float(logpZO_arr[0])

    return dict(base_prob=p, base_ent=ent, mc_var=mc_var, ens_var=ens_var, dens_logpZO=dens_val)


# Load and resize andrea.png
andrea_img = np.array(Image.open("media/andrea.png").convert("RGB").resize((256, 256)))
andrea_result = _infer_single_image(andrea_img)

# Eval dataset UQ stats (max and mean) per method
eval_eps_subset = [eval_episodes[i] for i in EVAL_EP_INDICES]

eval_ents, eval_mc_vars, eval_ens_vars, eval_dens = [], [], [], []
for ep in eval_eps_subset:
    _, _, e = run_inference_episode(baseline_model, ep, mode="baseline", device=DEVICE)
    eval_ents.extend(e)
    _, v, _ = run_inference_episode(mc_model, ep, mode="mc_dropout", device=DEVICE)
    eval_mc_vars.extend(v)
    _, v, _ = run_inference_episode(ensemble_model, ep, mode="ensemble", device=DEVICE)
    eval_ens_vars.extend(v)
    logpZO = predict_density_episode(density_model, dino_encoder, ep, DEVICE)
    eval_dens.extend(logpZO)

# Eval mean and 0.95 quantile per method
eval_ents_a, eval_mc_a, eval_ens_a, eval_dens_a = np.array(eval_ents), np.array(eval_mc_vars), np.array(eval_ens_vars), np.array(eval_dens)
eval_stats = {
    "Entropy (baseline)":  (eval_ents_a.mean(), np.percentile(eval_ents_a, 95)),
    "Variance (MC)":       (eval_mc_a.mean(), np.percentile(eval_mc_a, 95)),
    "Variance (Ensemble)": (eval_ens_a.mean(), np.percentile(eval_ens_a, 95)),
    "Density logpZO":      (eval_dens_a.mean(), np.percentile(eval_dens_a, 95)),
}
andrea_vals = {
    "Entropy (baseline)":  andrea_result["base_ent"],
    "Variance (MC)":       andrea_result["mc_var"],
    "Variance (Ensemble)": andrea_result["ens_var"],
    "Density logpZO":      andrea_result["dens_logpZO"],
}
method_names = list(andrea_vals.keys())

fig, axes = plt.subplots(1, 5, figsize=(14, 4))
axes[0].imshow(andrea_img)
axes[0].set_title(f"andrea.png\nP(fail)={andrea_result['base_prob']:.3f}", fontsize=9)
axes[0].axis("off")

for j, m in enumerate(method_names):
    ax = axes[j + 1]
    vals = [andrea_vals[m], eval_stats[m][0], eval_stats[m][1]]
    bars = ax.bar([0, 1, 2], vals, color=["coral", "steelblue", "seagreen"], alpha=0.9)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["input", "eval mean", "eval .95"], fontsize=7)
    ax.set_ylabel(m, fontsize=8)
    ax.set_ylim(0, max(vals) * 1.15 if max(vals) > 0 else 1)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01 * ax.get_ylim()[1], f"{v:.3g}", ha="center", va="bottom", fontsize=7)
fig.tight_layout()
plt.show()
print("andrea:", andrea_vals)
print("eval mean / 0.95:", {m: (eval_stats[m][0], eval_stats[m][1]) for m in method_names})

How about junwon? How about skittles spilling?

In [ ]:
# ── 4.1b  Same inference: junwon.png (single view) + skittles (two views) ─────
def _infer_two_views(front_np, wrist_np):
    """Run all UQ methods on one (front, wrist) pair; (256,256,3) uint8 RGB."""
    front_t = _img_transform(front_np).unsqueeze(0).to(DEVICE)
    wrist_t = _img_transform(wrist_np).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logit = baseline_model(front_t, wrist_t)
        p = torch.sigmoid(logit).item()
        ent = -(p * np.log(p + 1e-8) + (1 - p) * np.log(1 - p + 1e-8))
        mc_logits = mc_dropout_predict(mc_model.members[0], front_t, wrist_t, T=50)
        mc_var = float(torch.sigmoid(mc_logits).cpu().numpy().squeeze().var())
        ens_p = torch.sigmoid(ensemble_model(front_t, wrist_t)).cpu().numpy().squeeze()
        ens_var = float(ens_p.var())
        mini_ep = {"front_cam": [front_np], "wrist_cam": [wrist_np], "failure": [0]}
        dens_val = float(predict_density_episode(density_model, dino_encoder, mini_ep, DEVICE)[0])
    return dict(base_prob=p, base_ent=ent, mc_var=mc_var, ens_var=ens_var, dens_logpZO=dens_val)

# Reuse eval_stats from previous cell
eval_stats_short = {
    "Entropy": eval_stats["Entropy (baseline)"],
    "MC Var":  eval_stats["Variance (MC)"],
    "Ens Var": eval_stats["Variance (Ensemble)"],
    "Density": eval_stats["Density logpZO"],
}
method_names_short = list(eval_stats_short.keys())

# 1) junwon.png
junwon_img = np.array(Image.open("media/junwon.png").convert("RGB").resize((256, 256)))
junwon_result = _infer_two_views(junwon_img, junwon_img)

# 2) skittles
skittles_3rd  = np.array(Image.open("media/skittles_3rd.png").convert("RGB").resize((256, 256)))
skittles_wrist = np.array(Image.open("media/skittles_wrist.png").convert("RGB").resize((256, 256)))
skittles_result = _infer_two_views(skittles_3rd, skittles_wrist)

def _plot_result(ax_img, axes_methods, img_or_imgs, result, title):
    if isinstance(img_or_imgs, list):
        ax_img.imshow(np.concatenate([img_or_imgs[0], np.ones((256, 8, 3), dtype=np.uint8) * 200, img_or_imgs[1]], axis=1))
    else:
        ax_img.imshow(img_or_imgs)
    ax_img.set_title(title, fontsize=9)
    ax_img.axis("off")
    input_vals = [result["base_ent"], result["mc_var"], result["ens_var"], result["dens_logpZO"]]
    for j, m in enumerate(method_names_short):
        ax = axes_methods[j]
        vals = [input_vals[j], eval_stats_short[m][0], eval_stats_short[m][1]]
        bars = ax.bar([0, 1, 2], vals, color=["coral", "steelblue", "seagreen"], alpha=0.9)
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(["input", "eval mean", "eval .95"], fontsize=6)
        ax.set_ylabel(m, fontsize=7)
        ax.set_ylim(0, max(vals) * 1.15 if max(vals) > 0 else 1)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01 * (ax.get_ylim()[1] or 1), f"{v:.3g}", ha="center", va="bottom", fontsize=6)

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
_plot_result(axes[0, 0], [axes[0, 1], axes[0, 2], axes[0, 3], axes[0, 4]], junwon_img,
             junwon_result, f"junwon.png\nP(fail)={junwon_result['base_prob']:.3f}")
_plot_result(axes[1, 0], [axes[1, 1], axes[1, 2], axes[1, 3], axes[1, 4]], [skittles_3rd, skittles_wrist],
             skittles_result, f"skittles_3rd + wrist\nP(fail)={skittles_result['base_prob']:.3f}")
fig.tight_layout()
plt.show()
print("junwon:  ", {k: f"{v:.5f}" for k, v in junwon_result.items()})
print("skittles:", {k: f"{v:.5f}" for k, v in skittles_result.items()})

### Let's run prediction on OOD trajectories.

The jenga block's color has changed.

In [ ]:
# ── 4.2  OOD trajectories: indices 0, 1, 2, 8, 10 ───────────
# OOD_DIR, OOD_EP_INDICES, load_ood_episode, ood_episodes are defined in the setup cell

for i in range(len(ood_episodes)):
    ep0 = ood_episodes[i]
    idx0 = OOD_EP_INDICES[i]
    T0 = len(ep0["failure"])
    save_dir = OUTPUT_DIR / "videos" / "ood"
    save_dir.mkdir(parents=True, exist_ok=True)

    base_probs, _, base_ents = run_inference_episode(baseline_model, ep0, mode="baseline", device=DEVICE)
    path_b = render_episode_video(ep0, idx0, base_probs, base_ents, uq_label="Entropy", mode="baseline", fps=10,
                                uq_ymax=MODE_UQ_YMAX["baseline"], output_path=save_dir / f"ood_traj{idx0:02d}_baseline_T{T0}.mp4")
    print(f"Baseline: {path_b}")
    display_video(path_b)

    mc_probs, mc_vars, _ = run_inference_episode(mc_model, ep0, mode="mc_dropout", device=DEVICE)
    path_mc = render_episode_video(ep0, idx0, mc_probs, mc_vars, uq_label="Variance", mode="mc_dropout", fps=10,
                                uq_ymax=MODE_UQ_YMAX["mc_dropout"], output_path=save_dir / f"ood_traj{idx0:02d}_mc_dropout_T{T0}.mp4")
    print(f"MC Dropout: {path_mc}")
    display_video(path_mc)

    ens_probs, ens_vars, _ = run_inference_episode(ensemble_model, ep0, mode="ensemble", device=DEVICE)
    path_ens = render_episode_video(ep0, idx0, ens_probs, ens_vars, uq_label="Variance", mode="ensemble", fps=10,
                                    uq_ymax=MODE_UQ_YMAX["ensemble"], output_path=save_dir / f"ood_traj{idx0:02d}_ensemble_T{T0}.mp4")
    print(f"Ensemble: {path_ens}")
    display_video(path_ens)

    logpZO_ep0 = predict_density_episode(density_model, dino_encoder, ep0, DEVICE)
    path_dens = render_episode_video_density(ep0, idx0, ens_probs, logpZO_ep0, ens_vars,
                                            uq_label="Variance", mode="ensemble", fps=10,
                                            uq_ymax=MODE_UQ_YMAX["ensemble"], logpZO_ymax=DENSITY_YMAX,
                                            output_path=save_dir / f"ood_traj{idx0:02d}_density_T{T0}.mp4")
    print(f"Density: {path_dens}")
    display_video(path_dens)

### Discussion: What Are We Actually Measuring?

Each UQ method provides a **noisy approximation** of the true posterior:

| Method | What it approximates | Limitations |
|--------|---------------------|-------------|
| **Entropy** | $H(p) = -p\log p - (1-p)\log(1-p) \approx1 - p_c(x)$ | Cannot distinguish aleatoric from epistemic; confident-but-wrong models have low entropy |
| **MC Dropout** | $\text{Var}_q[f(x)]$ under dropout posterior $q(\theta)$ |  Quality depends on dropout rate; crude variational approximation |
| **Ensemble** | $\text{Var}_{k}[f_k(x)]$ across $K$ members |  $K\times$ compute and memory; member diversity limited by shared architecture |
| **Density** | $\log p_\phi(z_f, z_w)$ in feature space | Not posterior uncertainty — measures "have I seen this input?" not "is my prediction reliable?" |

**Important caveats:**
- These are all *approximations* — the true Bayesian posterior is intractable for neural networks.
- Bias != Variance. The model can be confident but wrong.
- **High uncertainty $\neq$ incorrect prediction.** A model can be uncertain about a correct prediction (e.g., an input near the boundary on the right side).
- **Low uncertainty $\neq$ correct prediction.** A model with systematic bias is confidently wrong — the robot acts, and the high confidence provides no warning.
- Density estimation measures something fundamentally different: it asks *"have I seen similar features before?"* rather than *"is my prediction reliable?"*

**Computational cost:**

| Method | Extra training | Inference cost |
|--------|---------------|---------------|
| Entropy (baseline) | None | $1\times$ forward |
| MC Dropout | None | $T\times$ forward ($T=50$) |
| Ensemble | $K\times$ training | $K\times$ forward ($K=7$) |
| Density | Separate flow model | $1\times$ DINOv2 + $1\times$ flow |

**The question:** How can we use these noisy uncertainty estimates **reliably** (with guarantees) for robotic decision-making?

---

## 5. Conformal Calibration for Uncertainty Thresholding

The uncertainty quantification (UQ) methods above produce a **scalar score** that reflects how uncertain the model is on a given input. However, to use this score in practice, we still need a principled way to choose a **threshold** for deciding whether a prediction should be trusted, rejected, or flagged for intervention. Selecting such a threshold manually is often ad hoc and provides no statistical guarantee.

**Conformal prediction** provides a distribution-free framework for calibrating this threshold using a held-out calibration dataset.

### Implementation

In conformal calibration, it is important to specify three things clearly:  
1. **the nonconformity score**,  
2. **the source of the calibration data**, and  
3. **the exact probabilistic guarantee provided by calibration**.

Given a held-out calibration dataset, we evaluate this nonconformity score on each calibration example and obtain a set of calibration scores
$$
s_i = u(x_i), \quad i=1,\dots,n.
$$
The uncertainty threshold is then calibrated by taking an upper quantile of these scores. Specifically, the conformal threshold is chosen as
$$
\hat{q}
=
\operatorname{Quantile}\!\left(
\{s_i\}_{i=1}^n;\;
\frac{\lceil (n+1)(1-\alpha)\rceil}{n}
\right),
$$
so that $\hat{q}$ corresponds to the $(1-\alpha)$-level quantile of the uncertainty scores on the calibration set, with the standard finite-sample conformal correction.


### Guarantee
When the calibration set is drawn from **in-distribution** data, the resulting threshold satisfies
$$
\Pr\!\left(u(X) \le \hat{q} \mid X \in \mathcal{D}_{\mathrm{in}}\right) \ge 1-\alpha.
$$

Equivalently,
$$
\Pr\!\left(u(X) > \hat{q} \mid X \in \mathcal{D}_{\mathrm{in}}\right) \le \alpha.
$$

This means that at least a fraction $1-\alpha$ of future in-distribution samples will have uncertainty below the calibrated threshold. In this sense, the procedure guarantees **coverage** over the target in-distribution population.

It is important to emphasize what this guarantee does **not** mean. This calibration procedure controls the rate at which **in-distribution** samples are assigned high uncertainty, so it can be interpreted as guaranteeing the **recall of in-distribution acceptance**. However, it does **not** directly guarantee the **precision of out-of-distribution detection**, nor does it guarantee that all out-of-distribution samples will have uncertainty above the threshold.

Therefore, conformal calibration gives a principled and finite-sample way to set the uncertainty threshold, but its guarantee is only with respect to the population on which calibration is performed.


### Decision Rule

Given a held-out calibration set, we compute the uncertainty scores on the calibration examples and use their conformal quantile to obtain a threshold $\hat{q}$. At test time, we then apply the decision rule
$$
\text{accept } x
\quad \text{if} \quad
u(x) \le \hat{q},
$$
and otherwise reject or flag the prediction as uncertain.

### 5.1 Calibration Dataset

In [ ]:
# Load calibration data
cal_episodes = load_episodes(["calibration_combined.pkl"])
print(f"Calibration episodes: {len(cal_episodes)}")

# Statistics
cal_steps = sum(len(ep["failure"]) for ep in cal_episodes)
cal_failures = sum(int((np.array(ep["failure"]) == 1).sum()) for ep in cal_episodes)
print(f"Total timesteps: {cal_steps}")
print(f"Failure timesteps: {cal_failures} ({100*cal_failures/cal_steps:.1f}%)")

### 5.2 Run Calibration

In [ ]:
# Conformal calibration — all 4 methods (+ density), with JSON caching
alpha = 0.1
CACHE_FILE = Path("output/conformal_thresholds.json")
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

classifier_cfg = [
    ("baseline",   "entropies"),
    ("mc_dropout", "variances"),
    ("ensemble",   "variances"),
]

if CACHE_FILE.exists():
    print(f"Loading cached thresholds from {CACHE_FILE}")
    with open(CACHE_FILE) as f:
        all_thresholds = json.load(f)
    all_cal_results = {}
else:
    model_objs = {"baseline": baseline_model, "mc_dropout": mc_model, "ensemble": ensemble_model}

    all_thresholds = {}
    all_cal_results = {}

    for mode, uq_metric in classifier_cfg:
        thr, cal_res = run_calibration(model_objs[mode], cal_episodes, mode=mode,
                                       device=DEVICE, alpha=alpha,
                                       uq_metric=uq_metric, batch_size=32)
        all_thresholds[mode] = {
            k: {"threshold": float(v["threshold"]),
                "n_samples": int(v["n_samples"]),
                "description": v["description"]}
            for k, v in thr.items() if k in ("image_id", "traj_id")
        }
        all_cal_results[mode] = cal_res

    # Density calibration
    dens_thr, dens_res = run_calibration_density(
        density_model, dino_encoder, cal_episodes, DEVICE, alpha=0.005)
    all_thresholds["density"] = {
        k: {"threshold": float(v["threshold"]),
            "n_samples": int(v["n_samples"]),
            "description": v["description"]}
        for k, v in dens_thr.items()
    }
    all_cal_results["density"] = dens_res

    with open(CACHE_FILE, "w") as f:
        _json_dump(all_thresholds, f, indent=2)
    print(f"Thresholds saved to {CACHE_FILE}")

print(f"\nConformal Calibration  (alpha={alpha}, target coverage >= {1-alpha:.0%})")
print("=" * 80)
for mode, uq_metric in classifier_cfg + [("density", "logpZO")]:
    print(f"\n  [{mode}]  UQ metric: {uq_metric}")
    for cal_key in ["image_id", "traj_id"]:
        info = all_thresholds[mode][cal_key]
        print(f"    {cal_key:12s} | tau={info['threshold']:.6f}  (n={info['n_samples']:5d})  {info['description']}")

# Convenience aliases for downstream cells
thresholds = all_thresholds["ensemble"]

### Visualizing the Conformal Threshold

To build intuition for conformal calibration, we plot the histogram of baseline entropy values on the calibration set and overlay the conformal threshold (image-level).

In [ ]:
# Histogram of baseline entropy on calibration set + conformal threshold
cal_entropies = []
for ep in cal_episodes:
    _, _, e = run_inference_episode(baseline_model, ep, mode="baseline", device=DEVICE)
    cal_entropies.extend(e)
cal_entropies = np.array(cal_entropies)

tau_baseline = all_thresholds["baseline"]["image_id"]["threshold"]

fig, ax = plt.subplots(figsize=(10, 4))
bins = np.arange(cal_entropies.min(), cal_entropies.max() + 0.001, 0.001)
ax.hist(cal_entropies, bins=bins, density=True, color="steelblue", alpha=0.7, edgecolor="white", label="Calibration entropy")
ax.axvline(tau_baseline, color="red", linestyle="--", linewidth=2,
           label=f"Conformal threshold (image_id) = {tau_baseline:.4f}")
ax.set_xlim(0.0, 0.02)
ax.set_xlabel("Entropy $H(p)$")
ax.set_ylabel("Density")
ax.set_title(f"Baseline Entropy Distribution on Calibration Set (alpha={alpha})")
ax.legend()
fig.tight_layout()
plt.show()

coverage = (cal_entropies <= tau_baseline).mean()
print(f"Empirical coverage: {coverage:.4f} (target: {1-alpha:.2f})")
print(f"Threshold: {tau_baseline:.6f}")
print(f"Entropy stats: mean={cal_entropies.mean():.4f}, median={np.median(cal_entropies):.4f}, "
      f"max={cal_entropies.max():.4f}")

### 5.3 Visualize with Threshold

normal data

In [ ]:
# Generate conformal threshold overlay videos for all 4 methods
ep = eval_episodes[1]

base_probs, base_vars, base_ents = run_inference_episode(baseline_model, ep, mode="baseline",   device=DEVICE)
mc_probs,   mc_vars,   _         = run_inference_episode(mc_model,       ep, mode="mc_dropout", device=DEVICE)
ens_probs,  ens_vars,  _         = run_inference_episode(ensemble_model, ep, mode="ensemble",   device=DEVICE)
logpZO = predict_density_episode(density_model, dino_encoder, ep, DEVICE)

clf_methods = [
    ("baseline",   base_probs, base_ents, "Entropy",  MODE_UQ_YMAX["baseline"]),
    ("mc_dropout", mc_probs,   mc_vars,   "Variance", MODE_UQ_YMAX["mc_dropout"]),
    ("ensemble",   ens_probs,  ens_vars,  "Variance", MODE_UQ_YMAX["ensemble"]),
]

for mode, probs, uq_scores, uq_label, ymax in clf_methods:
    tau = all_thresholds[mode]["image_id"]["threshold"]
    out = OUTPUT_DIR / "videos" / f"{mode}_cp_imageid_ep0.mp4"
    vp = render_episode_video(ep, 0, probs, uq_scores,
                              uq_label=uq_label, mode=mode, fps=10,
                              threshold=tau, uq_ymax=ymax, output_path=out)
    print(f"[{mode}]  tau(image_id)={tau:.6f}  ->  {vp}")
    display_video(vp)

# Density
tau_dens = all_thresholds["density"]["image_id"]["threshold"]
out_dens = OUTPUT_DIR / "videos" / "density_cp_imageid_ep0.mp4"
vp = render_episode_video_density(ep, 0, ens_probs, logpZO,
                                  classifier_uq=logpZO,
                                  uq_label="logpZO", mode="density", fps=10,
                                  threshold=tau_dens,
                                  logpZO_ymax=DENSITY_YMAX, output_path=out_dens)
print(f"[density]  tau(image_id)={tau_dens:.6f}  ->  {vp}")
display_video(vp)

ood data

In [ ]:
# Visualize conformal thresholds on OOD trajectories
ep = ood_episodes[0]
ep_idx = OOD_EP_INDICES[0]

base_probs, base_vars, base_ents = run_inference_episode(baseline_model, ep, mode="baseline",   device=DEVICE)
mc_probs,   mc_vars,   _         = run_inference_episode(mc_model,       ep, mode="mc_dropout", device=DEVICE)
ens_probs,  ens_vars,  _         = run_inference_episode(ensemble_model, ep, mode="ensemble",   device=DEVICE)
logpZO = predict_density_episode(density_model, dino_encoder, ep, DEVICE)

clf_methods = [
    ("baseline",   base_probs, base_ents, "Entropy",  MODE_UQ_YMAX["baseline"]),
    ("mc_dropout", mc_probs,   mc_vars,   "Variance", MODE_UQ_YMAX["mc_dropout"]),
    ("ensemble",   ens_probs,  ens_vars,  "Variance", MODE_UQ_YMAX["ensemble"]),
]

for mode, probs, uq_scores, uq_label, ymax in clf_methods:
    tau = all_thresholds[mode]["image_id"]["threshold"]
    out = OUTPUT_DIR / "videos" / f"{mode}_cp_imageid_ood{ep_idx}.mp4"
    vp = render_episode_video(ep, ep_idx, probs, uq_scores,
                              uq_label=uq_label, mode=mode, fps=10,
                              threshold=tau, uq_ymax=ymax, output_path=out)
    print(f"[{mode}]  tau(image_id)={tau:.6f}  ->  {vp}")
    display_video(vp)

tau_dens = all_thresholds["density"]["image_id"]["threshold"]
out_dens = OUTPUT_DIR / "videos" / f"density_cp_imageid_ood{ep_idx}.mp4"
vp = render_episode_video_density(ep, ep_idx, ens_probs, logpZO,
                                  classifier_uq=logpZO,
                                  uq_label="logpZO", mode="density", fps=10,
                                  threshold=tau_dens,
                                  logpZO_ymax=DENSITY_YMAX, output_path=out_dens)
print(f"[density]  tau(image_id)={tau_dens:.6f}  ->  {vp}")
display_video(vp)

### 5.5 Discussion: Limitations

**What does conformal prediction guarantee here?**

The guarantee $P(u < \tau) \geq 1-\alpha$ means: for a randomly drawn in-distribution sample, the UQ score is below $\tau$ with probability $\geq 1-\alpha$.


**This is NOT detection OOD (to be precise).** Conformal prediction guarantees **recall of in-distribution data** (low UQ for most calibration-like inputs), not detection of out-of-distribution inputs. An OOD sample *might* still have low UQ.

**Exchangeability violation frequently occurs:** Within a trajectory, timesteps are temporally correlated — they are NOT i.i.d. The image-level conformal guarantee is technically violated. This motivates trajectory-level conformal prediction (next section).

Formally, conformal prediction requires:
$$(X_1, Y_1), \ldots, (X_n, Y_n), (X_{n+1}, Y_{n+1}) \text{ are exchangeable}$$

Timesteps within a single trajectory violate this since $x_t$ depends on $x_{t-1}$.

**This is only a marginal guarantee.** The probability in the conformal guarantee is taken over both the sampling of the calibration set and the test sample. Thus, standard split conformal calibration guarantees coverage only on average, and does not ensure uniform performance for every subgroup, context, or trajectory prefix.

**Dataset-conditional guarantee.** A stronger statement conditions on the realized calibration dataset itself. In that case, one can ask for a guarantee of the form
$$
\Pr\!\left(u(X_{\mathrm{test}}) \le \tau \;\middle|\; \mathcal{D}_{\mathrm{cal}}\right)
\geq
\mathrm{Beta}^{-1}_{N+1-v,\;v}(\delta),
$$
where
$$
v := \left\lfloor (N+1)\hat{\alpha} \right\rfloor,
$$
$N$ is the size of the calibration set, $\hat{\alpha}$ is the calibration level, and $\mathrm{Beta}^{-1}_{a,b}(\delta)$ denotes the $\delta$-quantile of a Beta distribution with parameters $(a,b)$. This means that, with probability at least $1-\delta$ over the draw of the calibration set, the realized coverage on future test samples is lower bounded by the corresponding Beta quantile. In contrast to the usual marginal guarantee, this provides a **dataset-conditional** view of validity, which is more informative when the threshold is fixed after calibration and then reused on future data.

---

## 6. Trajectory-Level Conformal Prediction

### Key Insight

While timesteps within a trajectory are dependent, **trajectories themselves are i.i.d.** (each episode is an independent rollout).

We can restore the exchangeability assumption by working at the trajectory level:

$$S(\tau) = \max_{t} s_t(\tau)$$

where $s_t$ is the per-timestep UQ score. Using $S(\tau)$ as the nonconformity score, we get:

$$P\!\big(\max_t u_t < \hat{q}\big) \geq 1 - \alpha$$

This guarantees that the UQ stays below threshold for the **entire trajectory** with high probability.

### Causal Reconstruction of the threshold at test time to inidividual timesteps.
A trajectory-level threshold can also be used as an image-level threshold because the trajectory score is defined as the maximum over all per-image uncertainty scores:
$$
S(\tau)=\max_t s_t(\tau).
$$
If a trajectory satisfies
$$
\max_t u_t < \hat q,
$$
then every individual timestep in that trajectory must also satisfy
$$
u_t < \hat q \qquad \forall t.
$$
Therefore, the same calibrated threshold $\hat q$ can be applied to each image. The important distinction is that the conformal guarantee is still **trajectory-wise**: it guarantees that, with probability at least $1-\alpha$, **all images in the trajectory simultaneously** have uncertainty below the threshold. So the image-level rule inherits the same threshold, but the formal guarantee comes from treating the whole trajectory as the exchangeable unit.

**References:**
- [KnowNo (Ren et al., 2023)](https://arxiv.org/abs/2307.01928)
- [UniSafe (Seo et al., 2025)](https://arxiv.org/abs/2505.00779)

In [ ]:
# Compare image-level vs trajectory-level thresholds
image_threshold = thresholds["image_id"]["threshold"]
traj_threshold = thresholds["traj_id"]["threshold"]

print(f"Image-level threshold (image_id): {image_threshold:.6f}")
print(f"Trajectory-level threshold (traj_id): {traj_threshold:.6f}")
print(f"Ratio: {traj_threshold / (image_threshold + 1e-10):.2f}x")
print()
print("The trajectory-level threshold is higher because it needs to cover")
print("the MAX uncertainty in each trajectory, not just individual timesteps.")

In [ ]:
# Trajectory-level (traj_id) threshold overlay videos for all 4 methods
ep = eval_episodes[1]

base_probs, base_vars, base_ents = run_inference_episode(baseline_model, ep, mode="baseline",   device=DEVICE)
mc_probs,   mc_vars,   _         = run_inference_episode(mc_model,       ep, mode="mc_dropout", device=DEVICE)
ens_probs,  ens_vars,  _         = run_inference_episode(ensemble_model, ep, mode="ensemble",   device=DEVICE)
logpZO = predict_density_episode(density_model, dino_encoder, ep, DEVICE)

clf_methods = [
    ("baseline",   base_probs, base_ents, "Entropy",  MODE_UQ_YMAX["baseline"]),
    ("mc_dropout", mc_probs,   mc_vars,   "Variance", MODE_UQ_YMAX["mc_dropout"]),
    ("ensemble",   ens_probs,  ens_vars,  "Variance", MODE_UQ_YMAX["ensemble"]),
]

for mode, probs, uq_scores, uq_label, ymax in clf_methods:
    tau = all_thresholds[mode]["traj_id"]["threshold"]
    out = OUTPUT_DIR / "videos" / f"{mode}_cp_trajid_ep0.mp4"
    vp = render_episode_video(ep, 0, probs, uq_scores,
                              uq_label=uq_label, mode=mode, fps=10,
                              threshold=tau, uq_ymax=ymax, output_path=out)
    print(f"[{mode}]  tau(traj_id)={tau:.6f}  ->  {vp}")
    display_video(vp)

tau_dens = all_thresholds["density"]["traj_id"]["threshold"]
out_dens = OUTPUT_DIR / "videos" / "density_cp_trajid_ep0.mp4"
vp = render_episode_video_density(ep, 0, ens_probs, logpZO,
                                  classifier_uq=logpZO,
                                  uq_label="logpZO", mode="density", fps=10,
                                  threshold=tau_dens,
                                  logpZO_ymax=DENSITY_YMAX, output_path=out_dens)
print(f"[density]  tau(traj_id)={tau_dens:.6f}  ->  {vp}")
display_video(vp)

In [ ]:
# OOD trajectories with trajectory-level (traj_id) conformal threshold
ep = ood_episodes[0]
ep_idx = OOD_EP_INDICES[0]

base_probs, base_vars, base_ents = run_inference_episode(baseline_model, ep, mode="baseline",   device=DEVICE)
mc_probs,   mc_vars,   _         = run_inference_episode(mc_model,       ep, mode="mc_dropout", device=DEVICE)
ens_probs,  ens_vars,  _         = run_inference_episode(ensemble_model, ep, mode="ensemble",   device=DEVICE)
logpZO = predict_density_episode(density_model, dino_encoder, ep, DEVICE)

clf_methods = [
    ("baseline",   base_probs, base_ents, "Entropy",  MODE_UQ_YMAX["baseline"]),
    ("mc_dropout", mc_probs,   mc_vars,   "Variance", MODE_UQ_YMAX["mc_dropout"]),
    ("ensemble",   ens_probs,  ens_vars,  "Variance", MODE_UQ_YMAX["ensemble"]),
]

for mode, probs, uq_scores, uq_label, ymax in clf_methods:
    tau = all_thresholds[mode]["traj_id"]["threshold"]
    out = OUTPUT_DIR / "videos" / f"{mode}_cp_trajid_ood{ep_idx}.mp4"
    vp = render_episode_video(ep, ep_idx, probs, uq_scores,
                              uq_label=uq_label, mode=mode, fps=10,
                              threshold=tau, uq_ymax=ymax, output_path=out)
    print(f"[{mode}]  tau(traj_id)={tau:.6f}  ->  {vp}")
    display_video(vp)

tau_dens = all_thresholds["density"]["traj_id"]["threshold"]
out_dens = OUTPUT_DIR / "videos" / f"density_cp_trajid_ood{ep_idx}.mp4"
vp = render_episode_video_density(ep, ep_idx, ens_probs, logpZO,
                                  classifier_uq=logpZO,
                                  uq_label="logpZO", mode="density", fps=10,
                                  threshold=tau_dens,
                                  logpZO_ymax=DENSITY_YMAX, output_path=out_dens)
print(f"[density]  tau(traj_id)={tau_dens:.6f}  ->  {vp}")
display_video(vp)

## 7. VLM-based Failure Classifier

A key strength of conformal prediction is that it works for **any black-box model** — no assumptions about architecture, training procedure, or internal representations are needed.

To demonstrate this, we use a **Vision-Language Model (VLM)** as a failure classifier:
- **Model:** Qwen2.5-VL-3B-Instruct
- **Input:** 2 camera images + text prompt asking about Jenga failure
- **UQ metric:** Token probability of "yes" (failure) vs "no" (safe)
  - $P(\text{failure}) = \text{softmax}(\text{logit}_{\text{yes}}, \text{logit}_{\text{no}})_0$
  - Entropy of this binary prediction serves as uncertainty

No fine-tuning is needed — the VLM uses its pre-trained knowledge to assess failure.

In [ ]:
# Load VLM (this may take a moment to download)
from vlm_classifier import load_vlm, predict_with_vlm, predict_vlm_episode, build_failure_prompt

print("Loading VLM...")
vlm_model, vlm_processor = load_vlm(device=DEVICE)
print("VLM loaded!")

# Show the prompt
print(f"\nPrompt: {build_failure_prompt()}")

In [ ]:
# Demo: single image prediction (eval episode + media images)
ep = eval_episodes[0]
t_demo = len(ep["failure"]) // 2

result = predict_with_vlm(vlm_model, vlm_processor,
                          ep["front_cam"][t_demo], ep["wrist_cam"][t_demo])
print(f"VLM prediction (eval episode 0, t={t_demo}):")
print(f"  P(failure) = {result['prob']:.4f}")
print(f"  Entropy    = {result['entropy']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(decompress_image(ep["front_cam"][t_demo]))
axes[0].set_title("Front")
axes[0].axis("off")
axes[1].imshow(decompress_image(ep["wrist_cam"][t_demo]))
axes[1].set_title("Wrist")
axes[1].axis("off")
fig.suptitle(f"VLM: P(fail)={result['prob']:.3f}, H={result['entropy']:.4f}")
fig.tight_layout()
plt.show()

# VLM on media images
media = Path("media")
def load_img(path):
    return np.array(Image.open(path).convert("RGB"))

cases = [
    ("andrea.png", "andrea.png"),
    ("junwon.png", "junwon.png"),
    ("skittles_3rd.png", "skittles_wrist.png"),
]
for name_a, name_b in cases:
    front = load_img(media / name_a)
    wrist = load_img(media / name_b)
    res = predict_with_vlm(vlm_model, vlm_processor, front, wrist)
    print(f"\n{name_a} + {name_b}:  P(failure)={res['prob']:.4f},  Entropy={res['entropy']:.4f}")
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    axes[0].imshow(front)
    axes[0].set_title(name_a)
    axes[0].axis("off")
    axes[1].imshow(wrist)
    axes[1].set_title(name_b)
    axes[1].axis("off")
    fig.suptitle(f"VLM: P(fail)={res['prob']:.3f}, H={res['entropy']:.4f}")
    fig.tight_layout()
    plt.show()

In [ ]:
# Render VLM trajectory videos on eval episodes
for i in [0, 1]:
    ep = eval_episodes[i]
    print(f"Running VLM on episode {i} (stride=1)...")
    vlm_probs, vlm_ents = predict_vlm_episode(vlm_model, vlm_processor, ep, DEVICE, stride=1)

    VIDEO_VLM = OUTPUT_DIR / "videos" / f"vlm_ep{i}.mp4"
    video_path = render_episode_video(
        ep, i, vlm_probs, vlm_probs,
        uq_label="P(failure)", mode="vlm", fps=10,
        uq_ymax=1.0,
        output_path=VIDEO_VLM)
    print(f"Video saved: {video_path}")
    display_video(video_path)

In [ ]:
# VLM trajectory videos on OOD dataset
for idx in OOD_EP_INDICES:
    ep = ood_episodes[OOD_EP_INDICES.index(idx)]
    print(f"Running VLM on OOD episode {idx} (stride=1)...")
    vlm_probs, vlm_ents = predict_vlm_episode(vlm_model, vlm_processor, ep, DEVICE, stride=1)
    out = OUTPUT_DIR / "videos" / f"vlm_ood{idx}.mp4"
    video_path = render_episode_video(
        ep, idx, vlm_probs, vlm_probs,
        uq_label="P(failure)", mode="vlm", fps=10,
        uq_ymax=1.0, output_path=out)
    print(f"Video saved: {video_path}")
    display_video(video_path)

### 7.1 Calibrate VLM with Conformal Prediction

We can apply the exact same conformal calibration protocol to the VLM.

In [ ]:
# Run VLM on calibration episodes (strided for speed)
print("Running VLM on calibration episodes...")
vlm_cal_results = []
for i, ep in enumerate(cal_episodes):
    vlm_p, vlm_e = predict_vlm_episode(
        vlm_model, vlm_processor, ep, DEVICE, stride=10)
    labels = np.array(ep["failure"], dtype=np.float32)
    labels[labels < 0] = 0.0
    vlm_cal_results.append({
        "probs": vlm_p, "variances": vlm_e,  # use entropy as "variance"
        "entropies": vlm_e, "labels": labels
    })
    print(f"  Episode {i+1}/{len(cal_episodes)}", end="\r")
print()

# Calibrate
from conformal import calibrate_image_id, calibrate_traj_id
vlm_thresh_img, n_img = calibrate_image_id(vlm_cal_results, 0.01, "entropies")
vlm_thresh_traj, n_traj = calibrate_traj_id(vlm_cal_results, 0.01, "entropies")

print(f"\nVLM Conformal Thresholds (Entropy) (alpha={0.01}):")
print(f"  image_id:  {vlm_thresh_img:.6f}  (n={n_img})")
print(f"  traj_id:   {vlm_thresh_traj:.6f}  (n={n_traj})")

What is the problem here? Why is it?